# Track-map outline spike — derive the Tempelhof circuit from GPS

**Goal:** decide whether a *GPS-derived SVG* track map is clean enough to build into the
Race-Day Companion. We take one car's 20 Hz GPS telemetry, project lat/lng to flat metres,
extract one lap, and emit an SVG `<path>` + the projection transform.

**Why GPS, not the circuit PDF:** the cars and the track come from the *same* coordinate
source, so the dots sit on the track by construction — no manual georeferencing.

If the plots below look like Tempelhof, we bake `track_outline.json` into the UI and plot
live car dots with the SAME transform. Run top-to-bottom in Colab Enterprise.

In [ ]:
%pip install -q gcsfs pyarrow pandas numpy matplotlib

In [ ]:
import json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import google.auth

_, PROJECT_ID = google.auth.default()
print("Project:", PROJECT_ID)

CAR = 13   # which car's lap to trace — any running car works (13 = Da Costa, runs all race)
TELEM_DIR = "gs://class-demo/formula-e/berlin_2024/r10/telemetry/"  # 20 Hz, partitioned by car_number

## 1. Load one car's GPS telemetry

In [ ]:
try:
    df = pd.read_parquet(TELEM_DIR, filters=[("car_number", "==", CAR)], engine="pyarrow")
    if df.empty:  # partition value might be a string
        df = pd.read_parquet(TELEM_DIR, filters=[("car_number", "==", str(CAR))], engine="pyarrow")
except Exception as e:
    print("partitioned read failed, listing blobs instead:", e)
    from google.cloud import storage
    cli = storage.Client(project=PROJECT_ID)
    bucket, prefix = "class-demo", "formula-e/berlin_2024/r10/telemetry/"
    blobs = [b.name for b in cli.list_blobs(bucket, prefix=prefix) if b.name.endswith(".parquet")]
    print(f"{len(blobs)} parquet blobs; sample:", blobs[:3])
    hit = [b for b in blobs if f"car_number={CAR}" in b or f"/{CAR}." in b or f"_{CAR}." in b]
    df = pd.concat([pd.read_parquet(f"gs://{bucket}/{b}") for b in (hit or blobs[:1])], ignore_index=True)

print("shape:", df.shape)
print("columns:", list(df.columns))
df.head(3)

## 2. Sort by time and pull the GPS columns
Telemetry fields (per the data dictionary): `tv_gps_lat`, `tv_gps_long`, `tv_gps_head`, `tv_speed`, ...

In [ ]:
tcol = next((c for c in df.columns if re.search(r"ts|time|utc|wall|elapsed", c, re.I)), None)
print("time column:", tcol)
if tcol:
    df = df.sort_values(tcol)

lat = df["tv_gps_lat"].to_numpy(float)
lng = df["tv_gps_long"].to_numpy(float)
ok = np.isfinite(lat) & np.isfinite(lng) & (lat != 0) & (lng != 0)
lat, lng = lat[ok], lng[ok]
print(f"{len(lat)} GPS points | lat {lat.min():.5f}..{lat.max():.5f} | lng {lng.min():.5f}..{lng.max():.5f}")

## 3. Project lat/lng → flat metres (local equirectangular)
Tempelhof is ~2 km and flat, so a local equirectangular projection about the mean latitude is
accurate to well under a metre. The live UI will use this SAME transform for car dots, so we
save its parameters.

In [ ]:
lat0, lng0 = float(lat.mean()), float(lng.mean())
M_PER_DEG_LAT = 110540.0
M_PER_DEG_LNG = 111320.0 * math.cos(math.radians(lat0))
x = (lng - lng0) * M_PER_DEG_LNG
y = (lat - lat0) * M_PER_DEG_LAT

plt.figure(figsize=(8, 8))
plt.plot(x, y, lw=0.3)
plt.gca().set_aspect("equal")
plt.title(f"All GPS points, car {CAR} (every lap overlaid = the circuit)")
plt.show()

## 4. Extract ONE clean lap
The laps overlay into the track shape above. For a single crisp closed path, start at the first
point and cut where the trace next returns near it (~one lap).

In [ ]:
p0x, p0y = x[0], y[0]
d0 = np.hypot(x - p0x, y - p0y)
left = int(np.argmax(d0 > 80)) if (d0 > 80).any() else 0          # leave the start
tail = d0[left:] < 40
ret = left + int(np.argmax(tail)) if tail.any() else len(x) - 1   # first return near start
if ret <= left:
    ret = len(x) - 1
lx, ly = x[left:ret + 1], y[left:ret + 1]
print(f"lap slice: {len(lx)} points (idx {left}..{ret})")

plt.figure(figsize=(8, 8))
plt.plot(lx, ly, lw=1.0)
plt.gca().set_aspect("equal")
plt.title(f"One lap, car {CAR}")
plt.show()

## 5. Build the SVG path + viewBox (and save the transform)
Downsample to a few hundred points, normalise into an SVG viewBox (y flipped — SVG y grows
downward), and save everything the live UI needs to place cars with the same transform.

In [ ]:
N = 400
idx = np.linspace(0, len(lx) - 1, min(N, len(lx))).astype(int)
sx, sy = lx[idx], ly[idx]

pad = 20.0
minx, maxx, miny, maxy = float(sx.min()), float(sx.max()), float(sy.min()), float(sy.max())
W, H = maxx - minx, maxy - miny

def to_svg(px, py):
    return (px - minx) + pad, (maxy - py) + pad   # flip y for screen coords

pts = [to_svg(px, py) for px, py in zip(sx, sy)]
path = "M " + " L ".join(f"{px:.1f} {py:.1f}" for px, py in pts) + " Z"
viewBox = f"0 0 {W + 2 * pad:.1f} {H + 2 * pad:.1f}"
print("viewBox:", viewBox)
print("path points:", len(pts))
print("path head:", path[:120], "...")

plt.figure(figsize=(8, 8))
plt.plot([p[0] for p in pts], [p[1] for p in pts])
plt.gca().invert_yaxis()
plt.gca().set_aspect("equal")
plt.title("SVG-space preview (how it will render in the UI)")
plt.show()

In [ ]:
out = {
    "race_id": "berlin_2024_r10",
    "viewBox": viewBox,
    "path": path,
    # The live UI projects each car's (lat,lng) with these, then maps to SVG via
    #   x = (lng-lng0)*m_per_deg_lng ; y = (lat-lat0)*m_per_deg_lat
    #   svg_x = (x-minx)+pad ; svg_y = (maxy-y)+pad
    "transform": {
        "lat0": lat0, "lng0": lng0,
        "m_per_deg_lat": M_PER_DEG_LAT, "m_per_deg_lng": M_PER_DEG_LNG,
        "minx": minx, "maxy": maxy, "pad": pad,
    },
}
with open("track_outline.json", "w") as f:
    json.dump(out, f, indent=2)
print("wrote track_outline.json")

## Verdict
If the "one lap" and "SVG-space" plots look like Tempelhof, we're good: bake `track_outline.json`
into the companion page and plot live car dots with `svg(project(lat,lng))`. If it's noisy
(GPS spikes, the pit lane bleeding in), try another `CAR`, smooth the lap, or take a mid-race lap
window instead of the first lap.